# Assignment 1 - Constraint Mapping EDA

This notebook loads the three Assignment 1 constraint mapping files, shows sample records, inspects column names and example values, counts unique values, and examines missing values plus practical anomalies. Generated tables are saved in `solution/assignment_1/execution_outputs/`.

In [ ]:
from collections import defaultdict
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root by locating the Assignment 1 data folder."""
    for candidate in [start, *start.parents]:
        data_dir = candidate / "data" / "Assignment 1 - constraint_mapping"
        if data_dir.exists():
            return candidate
    raise FileNotFoundError("Could not find data/Assignment 1 - constraint_mapping from the current path.")


repo_root = find_repo_root()
data_dir = repo_root / "data" / "Assignment 1 - constraint_mapping"
execution_outputs_dir = repo_root / "solution" / "assignment_1" / "execution_outputs"
execution_outputs_dir.mkdir(parents=True, exist_ok=True)

constraint_files = {
    "dayzer": data_dir / "Dayzer PJMISO constraint list.csv",
    "market": data_dir / "Market PJMISO constraint list.csv",
    "pano": data_dir / "Pano PJMISO constraint list.csv",
}

{
    "data_dir": data_dir,
    "execution_outputs_dir": execution_outputs_dir,
    "constraint_files": constraint_files,
}

In [ ]:
constraints = {name: pd.read_csv(path) for name, path in constraint_files.items()}

for name, df in constraints.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

## Column Names and Data Types

In [ ]:
column_summary = pd.concat(
    [
        pd.DataFrame(
            {
                "file": name,
                "column": df.columns,
                "dtype": [df[col].dtype for col in df.columns],
                "non_null_count": [df[col].notna().sum() for col in df.columns],
                "example_value": [df[col].dropna().iloc[0] if df[col].notna().any() else pd.NA for col in df.columns],
            }
        )
        for name, df in constraints.items()
    ],
    ignore_index=True,
)

column_summary_path = execution_outputs_dir / "constraint_column_summary.csv"
column_summary.to_csv(column_summary_path, index=False)
print(f"Saved column summary to {column_summary_path}")

column_summary

## Example Rows

In [ ]:
sample_tables = []

for name, df in constraints.items():
    example_rows = df.head(5).copy()
    example_path = execution_outputs_dir / f"{name}_constraint_examples.csv"
    example_rows.to_csv(example_path, index=False)

    tagged_examples = example_rows.copy()
    tagged_examples.insert(0, "dataset", name)
    sample_tables.append(tagged_examples)

    print(f"\n{name.upper()} example rows saved to {example_path}")
    display(example_rows)

combined_samples = pd.concat(sample_tables, ignore_index=True, sort=False)
combined_samples_path = execution_outputs_dir / "constraint_sample_rows.csv"
combined_samples.to_csv(combined_samples_path, index=False)
print(f"\nSaved combined sample rows to {combined_samples_path}")


## Unique Value Counts

In [ ]:
unique_value_counts = pd.concat(
    [
        pd.DataFrame(
            {
                "dataset": name,
                "column": df.columns,
                "row_count": len(df),
                "non_null_count": [df[col].notna().sum() for col in df.columns],
                "unique_count_including_null": [df[col].nunique(dropna=False) for col in df.columns],
                "unique_count_excluding_null": [df[col].nunique(dropna=True) for col in df.columns],
            }
        )
        for name, df in constraints.items()
    ],
    ignore_index=True,
)

unique_counts_path = execution_outputs_dir / "constraint_unique_value_counts.csv"
unique_value_counts.to_csv(unique_counts_path, index=False)
print(f"Saved unique value counts to {unique_counts_path}")

unique_value_counts


## Missing Values

In [ ]:
missing_value_summary = pd.concat(
    [
        pd.DataFrame(
            {
                "dataset": name,
                "column": df.columns,
                "row_count": len(df),
                "missing_count": [df[col].isna().sum() for col in df.columns],
                "missing_percent": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
                "blank_string_count": [df[col].astype("string").str.strip().eq("").sum() for col in df.columns],
            }
        )
        for name, df in constraints.items()
    ],
    ignore_index=True,
)

missing_values_path = execution_outputs_dir / "constraint_missing_value_summary.csv"
missing_value_summary.to_csv(missing_values_path, index=False)
print(f"Saved missing-value summary to {missing_values_path}")

missing_value_summary


## Anomaly Checks

In [ ]:
anomaly_records = []

for name, df in constraints.items():
    anomaly_records.append(
        {
            "dataset": name,
            "check": "duplicate_full_rows",
            "value": int(df.duplicated().sum()),
            "notes": "Rows duplicated across all columns",
        }
    )

    for col in df.columns:
        if col.upper().endswith("ID") or col.upper() in {"CID", "PID"}:
            anomaly_records.append(
                {
                    "dataset": name,
                    "check": f"duplicate_identifier_{col}",
                    "value": int(df[col].duplicated().sum()),
                    "notes": "Duplicate values in an identifier-like column",
                }
            )

    text_columns = df.select_dtypes(include=["object", "string"]).columns
    for col in text_columns:
        stripped = df[col].astype("string").str.strip()
        anomaly_records.append(
            {
                "dataset": name,
                "check": f"leading_or_trailing_spaces_{col}",
                "value": int((df[col].astype("string") != stripped).sum()),
                "notes": "Text values that change after stripping whitespace",
            }
        )

    for col in ["Earliest", "Latest"]:
        if col in df.columns:
            parsed_dates = pd.to_datetime(df[col], errors="coerce", utc=True)
            anomaly_records.append(
                {
                    "dataset": name,
                    "check": f"unparseable_datetime_{col}",
                    "value": int(parsed_dates.isna().sum()),
                    "notes": "Date/time values that could not be parsed",
                }
            )

    if {"Earliest", "Latest"}.issubset(df.columns):
        earliest = pd.to_datetime(df["Earliest"], errors="coerce", utc=True)
        latest = pd.to_datetime(df["Latest"], errors="coerce", utc=True)
        anomaly_records.append(
            {
                "dataset": name,
                "check": "latest_before_earliest",
                "value": int((latest < earliest).sum()),
                "notes": "Rows where Latest occurs before Earliest",
            }
        )

anomaly_summary = pd.DataFrame(anomaly_records)
anomaly_summary_path = execution_outputs_dir / "constraint_anomaly_summary.csv"
anomaly_summary.to_csv(anomaly_summary_path, index=False)
print(f"Saved anomaly summary to {anomaly_summary_path}")

anomaly_summary


## Constraint Parsing and Normalization

Split each constraint record into facility text, contingency text, voltage/kV, extra tokens, utility/zone tokens, facility tokens, contingency tokens, and interface keywords. Normalize strings by lowercasing, replacing separators with spaces, removing punctuation, and standardizing terms such as `kv`, `xfmer`, and `ser dev`.

In [ ]:
SEPARATORS_PATTERN = re.compile(r"[-_./]+")
PUNCTUATION_PATTERN = re.compile(r"[^a-z0-9\s]")
VOLTAGE_PATTERN = re.compile(r"(?<!\d)(\d{2,4})\s*k\s*v\b|(?<!\d)(\d{2,4})kv\b")
STOP_TOKENS = {"", "a", "b", "c", "the", "and", "or", "to", "from", "kv", "line", "l"}
INTERFACE_KEYWORDS = {"interface", "import", "export", "flowgate", "flow", "tie", "path"}


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).lower()
    text = SEPARATORS_PATTERN.sub(" ", text)
    text = re.sub(r"\bk\s*v\b", "kv", text)
    text = re.sub(r"\bxfmer\b", "transformer", text)
    text = re.sub(r"\bser\s+dev\b", "series device", text)
    text = PUNCTUATION_PATTERN.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_voltage(*values):
    text = normalize_text(" ".join("" if pd.isna(value) else str(value) for value in values))
    match = VOLTAGE_PATTERN.search(text)
    if not match:
        return pd.NA
    return int(next(group for group in match.groups() if group))


def tokenize(text):
    return [token for token in normalize_text(text).split() if token not in STOP_TOKENS and not token.isdigit()]


def unique_tokens(tokens):
    return sorted(set(tokens))


def extract_interface_keywords(tokens):
    return sorted(token for token in set(tokens) if token in INTERFACE_KEYWORDS or token.endswith("interface"))


def extract_utility_zone_tokens(zone_value, raw_text):
    tokens = []
    if pd.notna(zone_value) and str(zone_value).strip():
        tokens.extend(tokenize(zone_value))
    tokens.extend(token.lower() for token in re.findall(r"\b[A-Z]{2,}\b", str(raw_text)))
    return unique_tokens(tokens)


def build_feature_frame(dataset_name, df):
    records = []
    for source_index, row in df.iterrows():
        if dataset_name == "market":
            facility_raw = " | ".join(str(row.get(col, "")) for col in ["CONSTRAINT", "REPORTEDNAME"] if pd.notna(row.get(col, pd.NA)))
            contingency_raw = row.get("CONTINGENCY", "")
            zone_raw = row.get("TOZONE", pd.NA)
            display_constraint = row.get("CONSTRAINT", "")
        elif dataset_name == "pano":
            facility_raw = row.get("Monitored Facility", "")
            contingency_raw = row.get("Contingency Name", "")
            zone_raw = pd.NA
            display_constraint = row.get("Monitored Facility", "")
        else:
            facility_raw = row.get("NAME", "")
            contingency_raw = ""
            zone_raw = pd.NA
            display_constraint = row.get("NAME", "")

        facility_tokens = unique_tokens(tokenize(facility_raw))
        contingency_tokens = unique_tokens(tokenize(contingency_raw))
        combined_text = f"{facility_raw} {contingency_raw}"
        all_tokens = unique_tokens([*facility_tokens, *contingency_tokens])
        records.append(
            {
                "dataset": dataset_name,
                "source_index": source_index,
                "display_constraint": display_constraint,
                "facility_raw": facility_raw,
                "contingency_raw": contingency_raw,
                "facility_normalized": normalize_text(facility_raw),
                "contingency_normalized": normalize_text(contingency_raw),
                "voltage_kv": extract_voltage(facility_raw, contingency_raw),
                "utility_zone_tokens": extract_utility_zone_tokens(zone_raw, combined_text),
                "facility_tokens": facility_tokens,
                "contingency_tokens": contingency_tokens,
                "interface_keywords": extract_interface_keywords(all_tokens),
                "extra_tokens": all_tokens,
            }
        )
    return pd.DataFrame(records)


feature_frames = {name: build_feature_frame(name, df) for name, df in constraints.items()}
parsed_features = pd.concat(feature_frames.values(), ignore_index=True)
parsed_features_export = parsed_features.copy()
for col in ["utility_zone_tokens", "facility_tokens", "contingency_tokens", "interface_keywords", "extra_tokens"]:
    parsed_features_export[col] = parsed_features_export[col].apply(lambda values: " | ".join(values))

parsed_features_path = execution_outputs_dir / "constraint_parsed_features.csv"
parsed_features_export.to_csv(parsed_features_path, index=False)
print(f"Saved parsed features to {parsed_features_path}")

parsed_features_export.head(10)


## Fuzzy Matching Against PJMISO Reference

Use the richest dataset, PJMISO/Market, as the reference. For each PJMISO record, choose the highest-scoring PANO match and the highest-scoring DAYZER match. PANO matching uses voltage when available plus overlapping facility and contingency tokens. DAYZER matching relies mostly on facility/interface similarity because DAYZER does not include contingency detail.

Score formula: `0.50 * facility_score + 0.25 * contingency_score + 0.15 * voltage_match + 0.10 * zone_overlap`.

In [ ]:
LOW_CONFIDENCE_THRESHOLD = 0.35
HIGH_CONFIDENCE_THRESHOLD = 0.70
MEDIUM_CONFIDENCE_THRESHOLD = 0.45
AMBIGUITY_MARGIN = 0.05
MAX_CANDIDATES = 400


def dice_similarity(left, right):
    left = set(left)
    right = set(right)
    if not left or not right:
        return 0.0
    return 2 * len(left & right) / (len(left) + len(right))


def voltage_match_score(left_voltage, right_voltage):
    if pd.isna(left_voltage) or pd.isna(right_voltage):
        return 0.0
    return 1.0 if int(left_voltage) == int(right_voltage) else 0.0


def score_match(reference, candidate):
    facility_score = dice_similarity(reference.facility_tokens, candidate.facility_tokens)
    contingency_score = dice_similarity(reference.contingency_tokens, candidate.contingency_tokens)
    voltage_match = voltage_match_score(reference.voltage_kv, candidate.voltage_kv)
    zone_overlap = dice_similarity(reference.utility_zone_tokens, candidate.utility_zone_tokens)
    total_score = (
        0.50 * facility_score
        + 0.25 * contingency_score
        + 0.15 * voltage_match
        + 0.10 * zone_overlap
    )
    return total_score, facility_score, contingency_score, voltage_match, zone_overlap


def confidence_label(score):
    if score >= HIGH_CONFIDENCE_THRESHOLD:
        return "high"
    if score >= MEDIUM_CONFIDENCE_THRESHOLD:
        return "medium"
    return "low"


def build_candidate_index(target_features, include_contingency=True):
    token_index = defaultdict(set)
    voltage_index = defaultdict(set)
    for row in target_features.itertuples():
        tokens = set(row.facility_tokens) | set(row.interface_keywords)
        if include_contingency:
            tokens |= set(row.contingency_tokens)
        for token in tokens:
            token_index[token].add(row.Index)
        if pd.notna(row.voltage_kv):
            voltage_index[int(row.voltage_kv)].add(row.Index)
    return token_index, voltage_index


def candidate_indices(reference, token_index, voltage_index, include_contingency=True):
    tokens = set(reference.facility_tokens) | set(reference.interface_keywords)
    if include_contingency:
        tokens |= set(reference.contingency_tokens)

    candidate_counts = defaultdict(int)
    for token in tokens:
        for idx in token_index.get(token, set()):
            candidate_counts[idx] += 1

    if pd.notna(reference.voltage_kv):
        for idx in voltage_index.get(int(reference.voltage_kv), set()):
            candidate_counts[idx] += 1

    ranked = sorted(candidate_counts, key=candidate_counts.get, reverse=True)
    return ranked[:MAX_CANDIDATES]


def best_match(reference, target_features, token_index, voltage_index, target_label, include_contingency=True):
    candidates = candidate_indices(reference, token_index, voltage_index, include_contingency=include_contingency)
    if not candidates:
        return {
            "match_source": target_label,
            "matched_index": pd.NA,
            "matched_constraint": pd.NA,
            "total_score": 0.0,
            "facility_score": 0.0,
            "contingency_score": 0.0,
            "voltage_match": 0.0,
            "zone_overlap": 0.0,
            "confidence": "low",
            "second_best_score": 0.0,
            "ambiguous": True,
        }

    scored = []
    for idx in candidates:
        candidate = target_features.loc[idx]
        total, facility, contingency, voltage, zone = score_match(reference, candidate)
        scored.append((total, facility, contingency, voltage, zone, idx, candidate.display_constraint))

    scored.sort(reverse=True, key=lambda item: item[0])
    best = scored[0]
    second_score = scored[1][0] if len(scored) > 1 else 0.0
    ambiguous = best[0] < LOW_CONFIDENCE_THRESHOLD or (best[0] - second_score) < AMBIGUITY_MARGIN
    return {
        "match_source": target_label,
        "matched_index": best[5],
        "matched_constraint": best[6],
        "total_score": round(best[0], 4),
        "facility_score": round(best[1], 4),
        "contingency_score": round(best[2], 4),
        "voltage_match": round(best[3], 4),
        "zone_overlap": round(best[4], 4),
        "confidence": confidence_label(best[0]),
        "second_best_score": round(second_score, 4),
        "ambiguous": ambiguous,
    }


market_features = feature_frames["market"].copy()
pano_features = feature_frames["pano"].copy()
dayzer_features = feature_frames["dayzer"].copy()

pano_token_index, pano_voltage_index = build_candidate_index(pano_features, include_contingency=True)
dayzer_token_index, dayzer_voltage_index = build_candidate_index(dayzer_features, include_contingency=False)

match_rows = []
support_rows = []
for reference in market_features.itertuples():
    pano_match = best_match(reference, pano_features, pano_token_index, pano_voltage_index, "pano", include_contingency=True)
    dayzer_match = best_match(reference, dayzer_features, dayzer_token_index, dayzer_voltage_index, "dayzer", include_contingency=False)

    match_rows.append(
        {
            "market_constraint": reference.display_constraint,
            "dayzer_constraint": dayzer_match["matched_constraint"],
            "pano_constraint": pano_match["matched_constraint"],
            "pano_confidence": pano_match["confidence"],
            "dayzer_confidence": dayzer_match["confidence"],
            "pano_score": pano_match["total_score"],
            "dayzer_score": dayzer_match["total_score"],
            "pano_ambiguous": pano_match["ambiguous"],
            "dayzer_ambiguous": dayzer_match["ambiguous"],
        }
    )

    for match in [pano_match, dayzer_match]:
        support_rows.append(
            {
                "market_constraint": reference.display_constraint,
                "match_source": match["match_source"],
                "matched_constraint": match["matched_constraint"],
                "confidence": match["confidence"],
                "total_score": match["total_score"],
                "facility_score": match["facility_score"],
                "contingency_score": match["contingency_score"],
                "voltage_match": match["voltage_match"],
                "zone_overlap": match["zone_overlap"],
                "second_best_score": match["second_best_score"],
                "ambiguous": match["ambiguous"],
            }
        )

constraint_matches = pd.DataFrame(match_rows)
constraint_match_support = pd.DataFrame(support_rows)

matches_path = execution_outputs_dir / "constraint_matches.csv"
support_path = execution_outputs_dir / "constraint_match_support.csv"
constraint_matches.to_csv(matches_path, index=False)
constraint_match_support.to_csv(support_path, index=False)
print(f"Saved constraint matches to {matches_path}")
print(f"Saved supporting match scores to {support_path}")

constraint_matches.head(10)


## Manual Review Samples

Create review sets for the top 20 highest-confidence matches, 20 medium-confidence matches, and 20 ambiguous matches.

In [ ]:
review_base = constraint_matches.copy()
review_base["average_score"] = review_base[["pano_score", "dayzer_score"]].mean(axis=1)
review_base["any_ambiguous"] = review_base[["pano_ambiguous", "dayzer_ambiguous"]].any(axis=1)

highest_confidence = review_base.sort_values("average_score", ascending=False).head(20).assign(review_bucket="highest_confidence")
medium_confidence = (
    review_base[review_base["average_score"].between(MEDIUM_CONFIDENCE_THRESHOLD, HIGH_CONFIDENCE_THRESHOLD, inclusive="left")]
    .sort_values("average_score", ascending=False)
    .head(20)
    .assign(review_bucket="medium_confidence")
)
ambiguous_matches = (
    review_base[review_base["any_ambiguous"]]
    .sort_values("average_score", ascending=False)
    .head(20)
    .assign(review_bucket="ambiguous")
)

manual_review_candidates = pd.concat(
    [highest_confidence, medium_confidence, ambiguous_matches],
    ignore_index=True,
)

manual_review_path = execution_outputs_dir / "constraint_manual_review_candidates.csv"
manual_review_candidates.to_csv(manual_review_path, index=False)
print(f"Saved manual review candidates to {manual_review_path}")

manual_review_candidates
